In [17]:
import pandas as pd
import numpy as np
import xgboost as xgb

from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from src.data_loader import load_men_data, load_women_data
from src.feature_engineering import *
from src.submission import generate_submission
from src.model import train_final_model, LEAN_PARAMS, LEAN_FEATURES, brier_score

In [9]:
data_dir = f'{project_root}/data'

# Men features
m_data = load_men_data(data_dir)
m_elo = compute_elo_ratings(m_data['MComSsn'])
m_seeds = parse_seeds(m_data['MTrnySeeds'])
m_stats = compute_season_stats(m_data['MDetSsn'])
massey = compute_massey_features(m_data['MOrdinals'])
m_conf = compute_conference_strength(m_data['MConf'], m_elo)
m_features = build_team_features(m_elo, m_seeds, m_stats, massey, m_conf)


In [19]:
m_matchups = build_matchup_matrix(m_data['MDetTrny'], m_features)
m_train_matchups = m_matchups[m_matchups['Season'] < 2022]
m_holdout_matchups = m_matchups[m_matchups['Season'] >= 2022]
MEN_TRAIN_FEATURES = [c for c in LEAN_FEATURES if c in m_train_matchups.columns]

m_holdout_model, m_holdout_feat_cols = train_final_model(
    m_train_matchups,
    feature_cols=MEN_TRAIN_FEATURES,
    xgb_params=LEAN_PARAMS,
    num_boost_round=200
)

In [20]:
m_dtest = xgb.DMatrix(m_holdout_matchups[m_holdout_feat_cols])
m_holdout_preds = m_holdout_model.predict(m_dtest)
print(f'Men holdout Brier (XGB only): {brier_score(m_holdout_matchups["Target"].values, m_holdout_preds): .5f}')

Men holdout Brier (XGB only):  0.19943


In [21]:
# Blend with Elo
from src.feature_engineering import elo_to_prob
m_elo_a = m_holdout_matchups.merge(m_elo, left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'])['Elo']
m_elo_b = m_holdout_matchups.merge(m_elo, left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'])['Elo']
m_elo_probs = elo_to_prob(m_elo_a.values, m_elo_b.values)

m_blended = 0.55 * m_holdout_preds + 0.45 * m_elo_probs
print(f'Men holdout Brier (blended):  {brier_score(m_holdout_matchups["Target"].values, m_blended):.5f}')
print(f'Men holdout Brier (pure Elo): {brier_score(m_holdout_matchups["Target"].values, m_elo_probs):.5f}')

Men holdout Brier (blended):  0.19551
Men holdout Brier (pure Elo): 0.20259


In [ ]:
# # Men training
# men_matchups = build_matchup_matrix(m_data['MDetTrny'], m_features)
# MEN_TRAIN_FEATURES = [c for c in LEAN_FEATURES if c in men_matchups.columns]
# m_model, m_feat_cols = train_final_model(
#     men_matchups, 
#     feature_cols = 
#     MEN_TRAIN_FEATURES, 
#     xgb_params=LEAN_PARAMS,
#     num_boost_round=200
# )

In [12]:
# Women features
w_data = load_women_data(data_dir)
w_elo = compute_elo_ratings(w_data['WComSsn'])
w_seeds = parse_seeds(w_data['WTrnySeeds'])
w_stats = compute_season_stats(w_data['WDetSsn'])
w_conf = compute_conference_strength(w_data['WConf'], w_elo)
w_features = build_team_features(w_elo, w_seeds, stats_df=w_stats, conf_df=w_conf)

WOMEN_LEAN_FEATURES = [
    'Elo_diff', 'SeedNum_diff', 'eFG_off_diff',
    'TO_rate_off_diff', 'OR_pct_diff', 'FT_rate_off_diff',
    'eFG_def_diff', 'TO_rate_def_diff', 'DR_pct_diff', 'FT_rate_def_diff',
    'Tempo_diff', 'PPG_diff', 'PPG_allowed_diff',
    'Win_pct_diff', 'Conf_Elo_mean_diff',
]

In [23]:
w_matchups = build_matchup_matrix(w_data['WDetTrny'], w_features)
w_train_matchups = w_matchups[w_matchups['Season'] < 2022]
w_holdout_matchups = w_matchups[w_matchups['Season'] >= 2022]
WOMEN_TRAIN_FEATURES = [c for c in WOMEN_LEAN_FEATURES if c in w_train_matchups.columns]

w_holdout_model, w_holdout_feat_cols = train_final_model(
    w_train_matchups,
    feature_cols=WOMEN_TRAIN_FEATURES,
    xgb_params=LEAN_PARAMS,
    num_boost_round=200
)

In [14]:
# # Women training
# w_matchups = build_matchup_matrix(w_data['WDetTrny'], w_features)
# WOMEN_TRAIN_FEATURES = [c for c in WOMEN_LEAN_FEATURES if c in w_matchups.columns]
# w_model, w_feat_cols = train_final_model(
#     w_matchups, 
#     feature_cols=WOMEN_TRAIN_FEATURES,
#     xgb_params=LEAN_PARAMS, 
#     num_boost_round=200
# )

In [15]:
all_elos = pd.concat([m_elo, w_elo], ignore_index=True)
elo_lookup = all_elos.set_index(['Season', 'TeamID'])['Elo'].to_dict()

In [24]:
w_dtest = xgb.DMatrix(w_holdout_matchups[w_holdout_feat_cols])
w_holdout_preds = w_holdout_model.predict(w_dtest)
print(f'Women holdout Brier (XGB only): {brier_score(w_holdout_matchups["Target"].values, w_holdout_preds): .5f}')

Women holdout Brier (XGB only):  0.14806


In [27]:
# Blend with Elo
from src.feature_engineering import elo_to_prob
w_elo_a = w_holdout_matchups.merge(w_elo, left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'])['Elo']
w_elo_b = w_holdout_matchups.merge(w_elo, left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'])['Elo']
w_elo_probs = elo_to_prob(w_elo_a.values, w_elo_b.values)

w_blended = 0.60 * w_holdout_preds + 0.40 * w_elo_probs
print(f'Women holdout Brier (blended):  {brier_score(w_holdout_matchups["Target"].values, w_blended):.5f}')
print(f'Women holdout Brier (pure Elo): {brier_score(w_holdout_matchups["Target"].values, w_elo_probs):.5f}')

Women holdout Brier (blended):  0.14390
Women holdout Brier (pure Elo): 0.15853


In [28]:
sub = generate_submission(
    '../data/SampleSubmissionStage1.csv',
    m_holdout_model, m_features, m_holdout_feat_cols,
    w_holdout_model, w_features, w_holdout_feat_cols,
    elo_lookup
)
sub.to_csv('../submissions/stage1_blended.csv', index=False)

Total matchups: 519144 (men: 261013, women: 258131)
XGBoost predictions: 519144, Elo fallback: 0
Pred range: [0.0200, 0.9800]
Pred mean: 0.4508
